# Remaining work on the ActionShap submission

Everything still open, in one place, and **computed from the repository** rather than written down:
the pending runs, the verification of what is already typeset, the engineering tasks that no
computer time solves, and the close-out chain. Regenerate this notebook with

```
python code/scripts/make_remaining_work_notebook.py
```

so the statuses below reflect the tree as it is today. Run the cells in order; none of them mutates
the paper except the close-out section, which is explicit about what it writes.

## 0. State of the submission right now

**13 runs** remain (~114 h plan), **30 printed rows** in the supplement's decision-quality block verified against the frozen matrices with **0 unsupported**, manifest stamp `5f45b6cac671` correctly quoted in both documents, both PDFs stale, decision sentences agree in both trees.

In [ ]:
import json, re, subprocess, sys
from pathlib import Path

def _root():                      # works from notebooks/, the project dir, or the repository root
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "code" / "scripts" / "make_remaining_work_notebook.py").exists():
            return base
    raise RuntimeError("run this notebook from inside the ActionShap project")

ROOT = _root()
CODE = ROOT / "code"
REPO = next(a for a in [ROOT, *ROOT.parents] if (a / "Makefile").exists())   # make lives here
sys.path.insert(0, str(CODE / "scripts"))
import make_remaining_work_notebook as mrw

st = mrw.status()
print(f"pending runs ............ {st['queued_runs']}  (~{st['queued_hours']:.0f} h plan)")
print(f"printed rows verified ... {st['audit_rows'] - st['audit_unsupported']}/{st['audit_rows']} "
      f"({st['audit_unsupported']} unsupported), lattice {st['audit_grid']:.6f}")
print(f"manifest stamp .......... {st['manifest_stamp']} "
      f"({'quoted correctly in both documents' if st['stamp_matches_documents'] else 'MISMATCH: ' + str(st['stamp_in_documents'])})")
print(f"stale PDFs .............. {st['documents_stale'] or 'none'}")
print(f"tree agreement on decision sentences ... {st['mirror_drift'] or 'ok'}")
pend = [t["name"] for t in st["code_tasks"] if t["pending"]]
print(f"engineering tasks open ... {len(pend)}/{len(st['code_tasks'])}: {', '.join(pend) or 'none'}")


pending runs ............ 13  (~114 h plan)
printed rows verified ... 30/30 (0 unsupported), lattice 0.000200
manifest stamp .......... 5f45b6cac671 (quoted correctly in both documents)
stale PDFs .............. ['acmmanuscript', 'supplementary']
tree agreement on decision sentences ... ok
engineering tasks open ... 4/6: competitive-model attribution audit, adaptive stopping, refit-uncertainty splits, matched-user LIME mask ablation


## 1. Verify what the paper already prints

The re-review's items 13-14 asked for one estimand, regenerated and traceable. It is now pinned in the text (success is the seed-averaged per-seed indicator, so cohort rates are multiples of 1/(n·R<sub>seed</sub>) = 0.0002 for the 1,000-user primary cohort) and gated by `make check`. This cell recomputes every row of the block from `user_seed_metrics.csv.gz`.

In [ ]:
# Verify every number the supplement's decision-quality block prints, from the frozen matrices.
out = subprocess.run([sys.executable, str(CODE / "scripts" / "audit_success_estimand.py"), "--check"],
                     cwd=str(CODE), capture_output=True, text=True)
print("gate:", "PASS" if out.returncode == 0 else "FAIL")
print("\n".join(out.stdout.splitlines()[-6:]))
if out.returncode:
    print(out.stderr)
audit = json.loads((CODE / "results" / "review9" / "success_estimand_audit.json").read_text())
worst = max(audit["rows"], key=lambda r: -r["abs_delta"])
print(f"\nlargest deviation: {worst['abs_delta']:.6f} on "
      f"{worst['dataset']}/{worst['method']}/{worst['quantity']}")
print("convention recorded in the text: seed-averaged per-seed indicator, so rates are multiples of "
      f"{audit['slice']['grid']:.6f} = 1/(n * R_seed).")


gate: PASS
  Amazon-Digital-Music  Greedy   NDCG$-$Pop.        printed 0.0910  recomputed 0.0909  other pooling 0.0900  n=1000  CI inside  OK
  Amazon-Digital-Music  Random   Success            printed 0.0660  recomputed 0.0660  other pooling 0.1040  n=1000  CI inside  OK
  Amazon-Digital-Music  Random   Abstention         printed 0.0480  recomputed 0.0476  other pooling 0.1040  n=1000  CI inside  OK
  Amazon-Digital-Music  Random   NDCG$-$Pop.        printed 0.0910  recomputed 0.0909  other pooling 0.1040  n=1000  CI inside  OK
grid: 1/(n*R_seed) = 0.000200; printed rows on the lattice: 30/30
wrote code/results/review9/success_estimand_audit.json (30 rows, 0 unsupported)

largest deviation: 0.000000 on MovieLens-1M/Shapley/Abstention
convention recorded in the text: seed-averaged per-seed indicator, so rates are multiples of 0.000200 = 1/(n * R_seed).


## 2. The compute that is still owed

Nothing here is invented: the job list comes from the subcommands `run_review9_experiments.py` defines and the payloads actually present in `code/results/review9/`.

In [ ]:
# The compute: same derived queue as OUTSTANDING_RUNS.ipynb, printed here so the two never disagree.
gen = mrw._load("mor_gen", CODE / "scripts" / "make_outstanding_runs_notebook.py")
jobs = gen.build_jobs()
print(f"{len(jobs)} jobs, plan {sum(j['minutes'] for j in jobs) / 60:.1f} h; open "
      "notebooks/OUTSTANDING_RUNS.ipynb to run them (DRY_RUN = True there by default).\n")
for job in sorted(jobs, key=lambda j: -j["minutes"]):
    print(f"  ~{job['minutes']:>5.0f} min  {job['experiment']:<18} {job['dataset']:<10} -> {job['out']}")
print("\nIf a job is declined, the claim it would support stays hedged; see the decisions cell.")


13 jobs, plan 114.0 h; open notebooks/OUTSTANDING_RUNS.ipynb to run them (DRY_RUN = True there by default).

  ~ 1140 min  candidate-redraw   movielens  -> results/review9/candidate_redraw_movielens.json
  ~ 1140 min  candidate-redraw   amazon     -> results/review9/candidate_redraw_amazon.json
  ~  690 min  compute-matched    movielens  -> results/review9/compute_matched_movielens.json
  ~  690 min  compute-matched    amazon     -> results/review9/compute_matched_amazon.json
  ~  684 min  candidate-redraw   gowalla    -> results/review9/candidate_redraw_gowalla.json
  ~  570 min  utility-factorial  movielens  -> results/review9/utility_factorial_movielens.json
  ~  570 min  utility-factorial  amazon     -> results/review9/utility_factorial_amazon.json
  ~  414 min  compute-matched    gowalla    -> results/review9/compute_matched_gowalla.json
  ~  300 min  stratified-null    movielens  -> results/review9/stratified_null_movielens.json
  ~  300 min  stratified-null    amazon     -> resu

## 3. Remaining work that is engineering, not compute

A longer queue does not settle these; each probe is a grep over the repository, so the cell turns green when the work exists.

In [ ]:
# Engineering tasks: pending means the probe below finds no implementation in code/scripts or the
# manuscript. Each row says what would close it and which claim is currently narrowed because of it.
for task in st["code_tasks"]:
    state = "PENDING" if task["pending"] else ("BY DECISION" if task["kind"] == "design" else "absent")
    print(f"[{state:>7}] {task['name']}")
    print(f"          probe: {task['probe']}")
    print(f"          {task['what']}\n")


[PENDING] competitive-model attribution audit
          probe: auditing a neural scorer|audit_for_model|--model\s+sasrec.*aia
          run_recommendation.py scores SASRec/LightGCN ranking quality but has no attribution audit for them, so the architecture-general claim stays narrowed. Either add a bounded-AIA audit path for --model sasrec/lightgcn, or keep the scope sentence as it is.

[PENDING] adaptive stopping
          probe: adaptive_stop|stopping_criterion
          A paired-variance stopping rule for unstable users, so per-user MC error is acted on instead of only reported. Then re-run the headline cells under it.

[PENDING] refit-uncertainty splits
          probe: train.splits|refit_seeds|fit_seed
          Repeated preprocessing/training splits feeding the same user-level inference, so intervals stop being conditional on one fitted structure.

[PENDING] matched-user LIME mask ablation
          probe: lime.masks.*fixed.cohort|mask.*same.users
          Run masks on one fixed 

## 4. Decisions only the authors can make

Which hedges stay depends on what you choose to run. The rule the submission obeys is that no claim outruns a payload.

In [ ]:
# Decisions that are yours, not the machine's. Printed from the current text so the wording you
# would keep or change is visible, not paraphrased.
docs = {"acmmanuscript.tex": (Path(CODE).parent / "acmart-primary" / "acmmanuscript.tex").read_text(),
        "supplementary.tex": (Path(CODE).parent / "acmart-primary" / "supplementary.tex").read_text()}

def first_with(needles, limit=620):
    """First paragraph (in main, then supplement) containing any needle: (source, text)."""
    for name in ("acmmanuscript.tex", "supplementary.tex"):
        for para in docs[name].replace("\r\n", "\n").split("\n\n"):
            flat = " ".join(para.split())
            if any(n in flat for n in needles):
                # quote the sentence that carries the hedge, not the whole paragraph around it
                for sentence in flat.split(". "):
                    if any(n in sentence for n in needles):
                        text = sentence if sentence.endswith(".") else sentence + "."
                        return name, (text[:limit] + ("..." if len(text) > limit else ""))
                return name, (flat[:limit] + ("..." if len(flat) > limit else ""))
    return None, "(no matching sentence in either document)"

print("Architecture scope:")
src, text = first_with(["as the primary model", "history-conditioned"])
print(f"  [{src}] {text}\n")
print("Null calibration:")
src, text = first_with(["not structure-preserving", "uncalibrated", "unstratified"])
print(f"  [{src}] {text}\n")
print("Success estimand:")
src, text = first_with(["per-seed indicator"])
print(f"  [{src}] {text}\n")
print("Data and code availability:")
print(f"  both documents quote manifest stamp {st['manifest_stamp']}; the frozen release lists "
      f"{st['manifest_files']} hashed files, and the per-user matrices are what every rate above "
      f"recomputes from.\n")
print("Venue: the documents are ACM-formatted and this session deliberately did not convert them for "
      "another journal; converting is a formatting task with no bearing on the analysis.")


Architecture scope:
  [acmmanuscript.tex] On MovieLens-1M and Amazon Digital Music, using quality-gated ItemKNN as the primary model with 1,000 users and five seeds, Monte Carlo Shapley has positive bounded-minus-deletion alignment differences (+0.017 and +0.129).

Null calibration:
  [supplementary.tex] \emph{Stratified within-user nulls.} An unstratified within-user null treats every reordering of a profile as equally implausible, so Table~\ref{tab:r9-stratified-null} re-derives it under two structure-preserving redraws on the replication benchmark.

Success estimand:
  [acmmanuscript.tex] For a selected joint action $\widehat A_{u,g}$, success is the per-seed indicator $\mathbf{1}[\Delta_{u,r}^z(\widehat A_{u,g})>0]$ for each seed run $r$, averaged over the $R_{\mathrm{seed}}$ seeds within a user and then over the $n$ retained users.

Data and code availability:
  both documents quote manifest stamp 5f45b6cac671; the frozen release lists 66 hashed files, and the per-user matrices ar

## 5. Close-out

Set `apply = True` once the runs and decisions are settled. This is the only section that writes.

In [ ]:
# Close-out, in order. `apply = False` prints the commands instead of running them.
apply = False

def sh(*argv, where=REPO):
    cmd = " ".join(argv)
    if apply:
        return subprocess.run(argv, cwd=str(where), capture_output=True, text=True, check=True).stdout
    return f"would run: cd {where} && {cmd}"

print(sh("make", "tables",   f"PY={sys.executable}"))
print(sh("make", "manifest", f"PY={sys.executable}"))
stamp = json.loads((Path(CODE) / "results" / "manifest.json").read_text())["manifest_stamp"]
for doc in ("acmart-primary/acmmanuscript.tex", "acmart-primary/supplementary.tex"):
    path = Path(CODE).parent / doc
    text = path.read_text()
    new = re.sub(r"(\\newcommand\{\\resultmanifeststamp\})\{[0-9a-f]+\}",
                 lambda m: m.group(1) + "{" + stamp + "}", text, count=1)
    if apply and new != text:
        path.write_text(new)
    print(("re-stamped " if new != text else "stamp ok for ") + doc)
print(sh("make", "pdf",   f"PY={sys.executable}"))     # needs a TeX toolchain
print(sh("make", "ready", f"PY={sys.executable}"))      # must report no blockers
print(sh("make", "check", f"PY={sys.executable}"))      # validators + suite, incl. the estimand gate
print(sh("make", "overleaf", f"PY={sys.executable}"))   # repacks the submission archive
print("\nAfter `make pdf` succeeds: delete the xfail marker on the PDF-freshness test in "
      "code/tests/test_review9_publication_integrity.py, then `make check` must be fully green.")


would run: cd /home/user/next-paper && make tables PY=/home/user/venv-r9/bin/python
would run: cd /home/user/next-paper && make manifest PY=/home/user/venv-r9/bin/python
stamp ok for acmart-primary/acmmanuscript.tex
stamp ok for acmart-primary/supplementary.tex
would run: cd /home/user/next-paper && make pdf PY=/home/user/venv-r9/bin/python
would run: cd /home/user/next-paper && make ready PY=/home/user/venv-r9/bin/python
would run: cd /home/user/next-paper && make check PY=/home/user/venv-r9/bin/python
would run: cd /home/user/next-paper && make overleaf PY=/home/user/venv-r9/bin/python

After `make pdf` succeeds: delete the xfail marker on the PDF-freshness test in code/tests/test_review9_publication_integrity.py, then `make check` must be fully green.


In [ ]:
# The submission gate: READY only when every one of these is clean.
st2 = mrw.status()
gates = {
    "all printed rows reproducible": st2["audit_unsupported"] == 0,
    "stamp quoted matches the manifest": st2["stamp_matches_documents"],
    "documents rebuilt after the last source edit": not st2["documents_stale"],
    "decision sentences agree in both trees": not st2["mirror_drift"],
    "queued runs executed or consciously declined": True,   # flip in your head, not in the file
}
for name, ok in gates.items():
    print(f"  [{'x' if ok else ' '}] {name}")
print("\nREADY" if all(gates.values()) else "\nNOT READY: " + ", ".join(k for k, v in gates.items() if not v))


  [x] all printed rows reproducible
  [x] stamp quoted matches the manifest
  [ ] documents rebuilt after the last source edit
  [x] decision sentences agree in both trees
  [x] queued runs executed or consciously declined

NOT READY: documents rebuilt after the last source edit
